# Angle Normalization (Gradient Descent / Least Squares)

This notebook calls the `functions/angle_normalization.py` module to perform angle normalization for nighttime light remote-sensing data.

**What it does**
- Read time-series data
- Correct the sensor zenith angle effect for each pixel
- Export corrected time series and evaluation metrics

In [ ]:
# Import the angle-normalization module
import sys
import os
import warnings
import matplotlib.pyplot as plt

# Configure fonts for notebook figures
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial Unicode MS', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# Add the project path so the functions package can be imported
cwd = os.getcwd()
project_root = cwd if os.path.basename(cwd) == 'ntl_prophet' else os.path.join(cwd, 'ntl_prophet')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Clear cached modules so the latest code is reloaded
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('functions'):
        del sys.modules[mod_name]

# Import functions used in this notebook
from functions.angle_normalization import (
    run_angle_normalization,
    display_comparison_results,
    plot_comparison_boxplot,
    plot_r2_distribution,
    readFile,
    visScatterAndFitCurve,
    visTimeSeries
)
from functions.timeseries_analysis import plot_mean_ntl_before_after

print('✅ Angle-normalization module loaded successfully')

## Parameters

In [ ]:
# ============== Configuration ==============
# Input and output paths
INPUT_DIR = r'.\processed'
OUTPUT_DIR = r'.\output'

# Dataset names to process (without the .txt suffix)
NAME_LIST = ['ntl_timeseries']

# Visualization options
PLOT_SCATTER = False   # Whether to plot scatter charts and fitted curves
PLOT_SERIES = False    # Whether to plot time-series line charts

# Parallel options
USE_PARALLEL = True    # Recommended for larger datasets
N_WORKERS = 16         # Number of worker processes; use None for CPU cores - 1

# Outlier filtering before normalization (recommended)
ENABLE_3SIGMA_FILTER = True
SIGMA_N = 3.0
NTL_UPPER_BOUND = 1000.0   # Empirical upper bound for this region; use None to disable
MIN_RECORDS_AFTER_FILTER = 10

print(f'Input directory: {INPUT_DIR}')
print(f'Output directory: {OUTPUT_DIR}')
print(f'Datasets to process: {NAME_LIST}')
print(f"Parallel processing: {'enabled' if USE_PARALLEL else 'disabled'}")
print(f"3-sigma filtering: {'enabled' if ENABLE_3SIGMA_FILTER else 'disabled'}, sigma={SIGMA_N}")
print(f"NTL upper bound: {NTL_UPPER_BOUND if NTL_UPPER_BOUND is not None else 'unlimited'}")

## Option 1: One-Click Run (Recommended)
Use `run_angle_normalization()` to complete the full workflow in one call.

In [ ]:
# Run angle normalization in one call, with optional parallel processing
for name in NAME_LIST:
    input_file = os.path.join(INPUT_DIR, f'{name}.txt')

    if not os.path.exists(input_file):
        print(f'⚠️ File does not exist: {input_file}')
        continue

    print(f'{"=" * 50}')
    print(f'Processing dataset: {name}')
    print(f'{"=" * 50}')

    results = run_angle_normalization(
        input_file=input_file,
        output_dir=OUTPUT_DIR,
        name=name,
        plot_scatter=PLOT_SCATTER,
        plot_series=PLOT_SERIES,
        verbose=False,
        show_progress=True,
        parallel=USE_PARALLEL,
        n_workers=N_WORKERS,
        prefilter_3sigma=ENABLE_3SIGMA_FILTER,
        prefilter_sigma=SIGMA_N,
        prefilter_ntl_upper_bound=NTL_UPPER_BOUND,
        min_records_after_filter=MIN_RECORDS_AFTER_FILTER
    )

    print('📁 Output files:')
    print(f'   Corrected series: {results["fit_result_path"]}')
    print(f'   Fit R²: {results["output_r2_path"]}')
    print(f'   Fit parameters: {results["output_params_path"]}')
    print(f'   Valid points: {results["num_points"]}')

    pf = results.get('prefilter_summary', {})
    if pf.get('enabled', False):
        print('   Outlier-filter summary:')
        print(f"      Points: {pf['points_before']} -> {pf['points_after']}")
        print(f"      Records: {pf['records_before']} -> {pf['records_after']}")
        print(f"      Removed records: {pf['records_removed']}")

    display_comparison_results(results, name, show_details=10)
    plot_comparison_boxplot(results, name)
    plot_r2_distribution(results, name)

    mean_ts = plot_mean_ntl_before_after(
        original_file=input_file,
        normalized_file=results['fit_result_path'],
        title=f'{name} mean nighttime light before vs after normalization'
    )
    print(f"   Dates in mean comparison: {mean_ts['stats']['n_dates']}")

print('✅ All processing completed!')